# 05. Подбор гиперпараметров

28 экспериментов: 12 ALS, 4 ItemKNN и 12 CatBoost. Все параметры выбираются по январю. Февраль здесь не используется. После поиска сравниваем исходный pipeline и лучший вариант поиска по январскому NDCG@10. Победителя используем в 06 и Streamlit.

Февральский test уже был просмотрен до этого поиска: это ограничение эксперимента сохраняется.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

if (Path.cwd() / "../movielens-reccomender/configs/experiment.json").exists():
    ROOT = (Path.cwd() / "../movielens-reccomender").resolve()
elif (Path.cwd() / "configs/experiment.json").exists():
    ROOT = Path.cwd()
else:
    ROOT = (Path.cwd() / "ml-1m/movielens-reccomender").resolve()
sys.path.insert(0, str(ROOT / "src"))
from movielens_recsys.tuning import search_spaces
from movielens_recsys.ranking import select_best_pipeline

REPORTS = ROOT / "reports/tuning"

## 1. Какие параметры проверяем

ALS: размер факторов, регуляризация и alpha. Число итераций фиксировано: 15.

ItemKNN: 20, 50, 100 или 200 соседей каждого фильма. Это не число кандидатов пользователя.

CatBoost: глубина, шаг обучения, регуляризация и random_strength. 11 случайных вариантов с seed=42 плюс исходные настройки. До 300 деревьев, early stopping=40.

In [2]:
als_grid, knn_grid, catboost_grid = search_spaces(seed=42)
print("ALS:", len(als_grid), "KNN:", len(knn_grid), "CatBoost:", len(catboost_grid))
display(pd.DataFrame(als_grid))
display(pd.DataFrame({"neighbors": knn_grid}))
display(pd.DataFrame(catboost_grid))

ALS: 12 KNN: 4 CatBoost: 12


,factors,regularization,alpha,iterations
0,32,0.05,1.0,15
1,32,0.05,10.0,15
2,32,0.10,1.0,15
3,32,0.10,10.0,15
4,64,0.05,1.0,15
5,64,0.05,10.0,15
6,64,0.10,1.0,15
7,64,0.10,10.0,15
8,128,0.05,1.0,15
9,128,0.05,10.0,15


,neighbors
0,20
1,50
2,100
3,200


,depth,learning_rate,l2_leaf_reg,random_strength
0,6,0.05,3,1.0
1,4,0.05,3,2.0
2,4,0.03,3,0.5
3,6,0.03,10,2.0
4,6,0.03,3,1.0
5,6,0.03,1,1.0
6,4,0.05,10,2.0
7,4,0.05,3,1.0
8,8,0.05,10,1.0
9,4,0.05,1,2.0


## 2. Протокол поиска

Последовательность: ALS → KNN → новые снимки сентября–декабря → CatBoost. Это поэтапный поиск, а не полный перебор ALS × KNN × CatBoost. Повторный поиск запускается из корня проекта командой `python scripts/tune_hyperparameters.py`. В файлы сохраняются только таблицы, которые читаются ниже, и выбранная экспериментальная модель с конфигом.

## 3. Результаты ALS

Выбираем наибольший Recall@150: столько кандидатов ALS передаёт в Union. Recall@250 сохраняем для сравнения с прошлым экспериментом.

In [3]:
als_results = pd.read_csv(REPORTS / "als_trials.csv")
display(als_results.sort_values("recall_at_150", ascending=False))

,trial,factors,regularization,alpha,iterations,recall_at_150,recall_at_250,ndcg_at_250
3,4,32,0.10,10.0,15,0.419330,0.534806,0.229341
1,2,32,0.05,10.0,15,0.418769,0.532322,0.228657
0,1,32,0.05,1.0,15,0.406996,0.513422,0.228514
2,3,32,0.10,1.0,15,0.405989,0.513445,0.228784
7,8,64,0.10,10.0,15,0.392016,0.498315,0.215639
5,6,64,0.05,10.0,15,0.391266,0.501893,0.215880
6,7,64,0.10,1.0,15,0.378142,0.484496,0.215766
4,5,64,0.05,1.0,15,0.377241,0.483474,0.215806
9,10,128,0.05,10.0,15,0.347004,0.455919,0.190365
11,12,128,0.10,10.0,15,0.346256,0.450354,0.190320


## 4. Результаты ItemKNN

Фиксируем выбранный ALS. Меняем число соседей фильма и выбираем по Recall@250 всего Union: KNN должен находить полезные дополнения к ALS.

In [4]:
knn_results = pd.read_csv(REPORTS / "knn_trials.csv")
display(knn_results.sort_values("union_recall_at_250", ascending=False))

,trial,neighbors,recall_at_100,recall_at_250,union_recall_at_250
0,1,20,0.284045,0.462814,0.534977
2,3,100,0.276277,0.445281,0.534620
3,4,200,0.265588,0.436108,0.534426
1,2,50,0.278133,0.460495,0.532274


## 5. Результаты CatBoost

Train-снимки пересчитаны с выбранными ALS/KNN. В fitting остаются группы с разными метками. В validation остаются все пользователи, включая промахи retrieval. Признаки — прежние 11.

Early stopping отслеживает NDCG на кандидатах. Для сравнения моделей считаем полный end-to-end NDCG@10 против всех положительных январских фильмов.

In [5]:
catboost_results = pd.read_csv(REPORTS / "catboost_trials.csv")
display(catboost_results.sort_values("ndcg", ascending=False))

,trial,depth,learning_rate,l2_leaf_reg,random_strength,tree_count,model,k,precision,recall,hitrate,ndcg,catalog_coverage,mean_list_length,users
7,8,4,0.05,3,1.0,163,CatBoost_tuned,10,0.107905,0.065845,0.416996,0.128946,0.056009,10.0,506
2,3,4,0.03,3,0.5,287,CatBoost_tuned,10,0.107708,0.064778,0.415020,0.127803,0.053834,10.0,506
5,6,6,0.03,1,1.0,227,CatBoost_tuned,10,0.105336,0.062448,0.407115,0.127530,0.053290,10.0,506
10,11,8,0.03,1,1.0,281,CatBoost_tuned,10,0.106522,0.064337,0.411067,0.127301,0.054105,10.0,506
6,7,4,0.05,10,2.0,286,CatBoost_tuned,10,0.107905,0.065649,0.416996,0.127181,0.053562,10.0,506
9,10,4,0.05,1,2.0,143,CatBoost_tuned,10,0.105534,0.064399,0.405138,0.127162,0.055465,10.0,506
11,12,4,0.03,3,1.0,183,CatBoost_tuned,10,0.106719,0.064987,0.415020,0.126443,0.054377,10.0,506
1,2,4,0.05,3,2.0,106,CatBoost_tuned,10,0.105534,0.064513,0.405138,0.125522,0.054921,10.0,506
0,1,6,0.05,3,1.0,79,CatBoost_tuned,10,0.103755,0.064546,0.413043,0.122952,0.052746,10.0,506
4,5,6,0.03,3,1.0,99,CatBoost_tuned,10,0.103755,0.062989,0.407115,0.122497,0.052474,10.0,506


## 6. Выбор модели для теста и приложения

Исходный CatBoost оценивается на исходных кандидатах; новый — на кандидатах выбранного retrieval. Даты, пользователи и полный ground truth одинаковые. Выбираем наибольший NDCG@10. При равенстве оставляем исходную модель. Нельзя брать CatBoost из одного варианта, а настройки ALS/KNN из другого.

Сейчас исходная модель лучше: 0.12999 против 0.12895. Поэтому 06 и приложение используют её. После нового поиска выбор автоматически обновится по этой же январской таблице.

In [6]:
comparison = pd.read_csv(REPORTS / "pipeline_comparison.csv")
display(comparison)
tuned_config = json.loads((ROOT / "configs/tuned_validation.json").read_text(encoding="utf-8"))
print("ALS:", tuned_config["retrieval"]["als_parameters"])
print("ItemKNN neighbors:", tuned_config["retrieval"]["item_knn_neighbors"])
print("CatBoost:", tuned_config["ranking"]["parameters"])

selected_config, selected_model_path = select_best_pipeline(ROOT)
print("Выбрано для 06 и Streamlit:", selected_model_path.name)
print("ALS:", selected_config["retrieval"]["als_parameters"])
print("ItemKNN neighbors:", selected_config["retrieval"]["item_knn_neighbors"])

,model,k,precision,recall,hitrate,ndcg,catalog_coverage,mean_list_length,users
0,CatBoost_original,10,0.106126,0.064781,0.413043,0.129987,0.060903,10.0,506
1,CatBoost_tuned,10,0.107905,0.065845,0.416996,0.128946,0.056009,10.0,506


ALS: {'factors': 32, 'regularization': 0.1, 'alpha': 10.0, 'iterations': 15}
ItemKNN neighbors: 20
CatBoost: {'depth': 4, 'learning_rate': 0.05, 'l2_leaf_reg': 3, 'random_strength': 1.0, 'iterations': 163, 'loss_function': 'YetiRank', 'eval_metric': 'NDCG:top=10', 'thread_count': 4, 'allow_writing_files': False}
Выбрано для 06 и Streamlit: catboost_base.cbm
ALS: {'factors': 32, 'regularization': 0.1, 'iterations': 15, 'alpha': 1.0}
ItemKNN neighbors: 50
